# 10. MAF でワークフローを構築する

## コードの解説

MAF の `SequentialBuilder` と `ConcurrentBuilder` を使って宣言的なワークフローを構築します。
複数のエージェントを組み合わせることで、複雑なビジネスロジックを実現できます。

### ワークフロービルダーの種類

| ビルダー | 実行パターン | 用途 |
|--------|-----------|------|
| `SequentialBuilder` | A → B → C（順次実行） | 前のエージェントの結果を次が使う処理 |
| `ConcurrentBuilder` | A, B, C を同時実行 | 独立したタスクを並列処理して高速化 |
| `WorkflowBuilder` | カスタム DAG | 条件分岐・複雑なルーティング |
| `MagenticBuilder` | AI によるオーケストレーション | 動的なエージェント選択 |

### Sequential（順次）ワークフロー

```
ユーザー質問
    ↓
ResearchAgent（情報収集）
    ↓  結果を引き継ぐ
SummaryAgent（要約・整理）
    ↓
最終回答
```

### Concurrent（並列）ワークフロー

```
ユーザー質問
    ├─→ FlightAgent（フライト調査）  ┐
    ├─→ HotelAgent（ホテル調査）    ├─ 結果を集約 → 最終回答
    └─→ ActivityAgent（観光調査）   ┘
       （すべて同時実行）
```

### このノートブックの構成

コードでは以下の 2 つのワークフローを順番に実行します：
1. **Sequential ワークフロー**：調査 → 要約の 2 段階処理
2. **Concurrent ワークフロー**：フライト・ホテル・観光を並列調査

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
API_VERSION           = os.getenv("AZURE_OPENAI_API_VERSION")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity.aio import DefaultAzureCredential
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())

# LLM クライアント（全エージェント共有）
chat_client = OpenAIChatCompletionClient(
    **auth_kwargs,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=API_VERSION,
    model=MODEL_DEPLOYMENT_NAME,
)


# ---------------------------------------------------------------
# ワークフロー 1: Sequential（順次）ワークフロー
#
# ResearchAgent が情報を収集し、その結果を SummaryAgent が要約する
# ポイント: SequentialBuilder(participants=[A, B, C]).build() で構築
# ---------------------------------------------------------------
research_agent = chat_client.as_agent(
    name="ResearchAgent",
    instructions=(
        "あなたは旅行情報の調査専門家です。"
        "指定された旅行先について、観光スポット・交通・気候などの情報を収集してください。"
        "簡潔な箇条書きで情報をまとめてください。"
    ),
)

summary_agent = chat_client.as_agent(
    name="SummaryAgent",
    instructions=(
        "あなたは情報整理の専門家です。"
        "前のエージェントが収集した情報をもとに、わかりやすい旅行ガイドを作成してください。"
        "初心者旅行者向けに、重要なポイントをわかりやすくまとめてください。"
    ),
)

sequential_workflow = SequentialBuilder(
    participants=[research_agent, summary_agent]
).build()

print("=== Sequential ワークフロー: 調査 → 要約 ===")
seq_result = await sequential_workflow.run("東京旅行について教えてください。")
for output in seq_result.get_outputs():
    for msg in output:
        print(msg.text)
print()


# ---------------------------------------------------------------
# ワークフロー 2: Concurrent（並列）ワークフロー
#
# フライト・ホテル・観光の 3 エージェントが同時に実行され、結果が集約される
# ポイント: ConcurrentBuilder(participants=[A, B, C]).build() で構築
# ---------------------------------------------------------------
flight_agent = chat_client.as_agent(
    name="FlightAgent",
    instructions="フライト予約のアドバイスを 2〜3 文で簡潔に提供してください。",
)

hotel_agent = chat_client.as_agent(
    name="HotelAgent",
    instructions="ホテルの選び方とおすすめエリアを 2〜3 文で簡潔に提供してください。",
)

activity_agent = chat_client.as_agent(
    name="ActivityAgent",
    instructions="おすすめ観光スポットとアクティビティを 2〜3 文で簡潔に提供してください。",
)

concurrent_workflow = ConcurrentBuilder(
    participants=[flight_agent, hotel_agent, activity_agent]
).build()

print("=== Concurrent ワークフロー: フライト・ホテル・観光を並列調査 ===")
conc_result = await concurrent_workflow.run("東京 3 泊 4 日の旅行を計画しています。")
for output in conc_result.get_outputs():
    for msg in output:
        print(msg.text)
